# Vision Transformer in Keras (Simple Version)

This notebook shows a simpler version of the Vision Transformer idea in Keras.

The goal is to understand the main steps:
- load a trained CNN,
- take feature maps from the CNN,
- turn them into tokens,
- add position information,
- apply transformer blocks,
- and train the final model.

This version keeps the explanation short and the code easier to read.

## 1. Import libraries

In [ ]:
import os
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

## 2. Set random seed

In [ ]:
seed = 7331
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

## 3. Load the pretrained CNN model

The CNN works as a feature extractor.

In [ ]:
CNN_PATH = os.path.join('.', 'ai-capstone-keras-best-model-model_downloaded.keras')
cnn_model = tf.keras.models.load_model(CNN_PATH)
cnn_model.trainable = False
cnn_model.summary()

## 4. Choose the CNN feature layer

We take the output of a middle layer to turn it into tokens.

In [ ]:
feature_layer_name = 'batch_normalization_5'
features = cnn_model.get_layer(feature_layer_name).output
print(features.shape)

## 5. Add positional embeddings

The transformer does not know the order of tokens by itself, so we add position information.

In [ ]:
@tf.keras.utils.register_keras_serializable(package='Custom')
class AddPositionEmbedding(layers.Layer):
    def __init__(self, num_patches, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.pos = self.add_weight(
            name='pos_embedding',
            shape=(1, num_patches, embed_dim),
            initializer='random_normal',
            trainable=True)

    def call(self, tokens):
        return tokens + self.pos

## 6. Define a transformer block

In [ ]:
@tf.keras.utils.register_keras_serializable(package='Custom')
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads=8, mlp_dim=2048, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.mha = layers.MultiHeadAttention(num_heads, key_dim=embed_dim)
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp = tf.keras.Sequential([
            layers.Dense(mlp_dim, activation='gelu'),
            layers.Dropout(dropout),
            layers.Dense(embed_dim),
            layers.Dropout(dropout),
        ])

    def call(self, x):
        x = self.norm1(x + self.mha(x, x))
        return self.norm2(x + self.mlp(x))

## 7. Build the hybrid CNN-ViT model

We flatten feature maps into tokens, add positions, pass them through transformer blocks, and classify the result.

In [ ]:
H, W, C = features.shape[1], features.shape[2], features.shape[3]

x = layers.Reshape((H * W, C))(features)
x = AddPositionEmbedding(H * W, C)(x)

for _ in range(4):
    x = TransformerBlock(C, num_heads=8, mlp_dim=2048)(x)

x = layers.GlobalAveragePooling1D()(x)
outputs = layers.Dense(2, activation='softmax')(x)

hybrid_model = Model(cnn_model.input, outputs, name='CNN_ViT_Hybrid')
hybrid_model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
                     loss='categorical_crossentropy',
                     metrics=['accuracy'])

hybrid_model.summary()

## 8. Prepare image data

Use augmentation and image generators for training and validation.

In [ ]:
DATASET_PATH = os.path.join('.', 'images_dataSAT')

img_w, img_h = 64, 64
batch_size = 4

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=30,
    horizontal_flip=True,
)

train_gen = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(img_w, img_h),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_gen = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(img_w, img_h),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=True
)

print(train_gen.class_indices)

## 9. Train the model

In [ ]:
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    'vision_transformer_simple.keras',
    monitor='val_loss',
    mode='min',
    save_best_only=True,
    verbose=1
)

history = hybrid_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=3,
    steps_per_epoch=128,
    callbacks=[checkpoint_cb],
    verbose=1
)

## 10. Check the training curve

In [ ]:
plt = tf.keras.utils.plot_model

# accuracy plot
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].plot(history.history['accuracy'], label='Train Accuracy')
axs[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axs[0].set_title('Accuracy')
axs[0].legend()

axs[1].plot(history.history['loss'], label='Train Loss')
axs[1].plot(history.history['val_loss'], label='Validation Loss')
axs[1].set_title('Loss')
axs[1].legend()

plt.tight_layout()
plt.show()

## Summary

This notebook shows the main idea behind a CNN-ViT hybrid model:
- CNN extracts important image features,
- transformer learns relationships between them,
- and the final layer predicts the class.

The method is useful because it combines local pattern learning from CNNs and global context learning from transformers.